# SpatialData Prototype: Labels + Nimbus Table for `SLIDE-0329_crop_2048`

This notebook extends the image-ingestion prototype to:
- load the cropped merged image with `sopa.io.ome_tif(...)`,
- import the whole-cell label mask,
- optionally import the nuclear label mask,
- load the Nimbus quantification table as a `SpatialData` table linked to `cell_labels`,
- aggregate image intensities over `cell_labels`,
- compare mask labels, Nimbus IDs, and aggregated-table IDs.

The notebook keeps two objects on purpose:
- `base_sdata`: image + labels + `nimbus_table`
- `agg_sdata`: `cell_labels` + `agg_table`

The goal here is simple, explicit validation of ID consistency. No shapes conversion, table merging, or production code changes in this notebook.


In [1]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

CROP_DIR = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048")
OUTPUT_DIR = CROP_DIR / "outputs"
FULL_MERGE_PATH = OUTPUT_DIR / "SLIDE-0329_crop_2048_full_merge.ome.tif"
CELL_MASK_PATH = OUTPUT_DIR / "masks" / "SLIDE-0329_crop_2048_whole_cell.tiff"
NUCLEAR_MASK_PATH = OUTPUT_DIR / "masks" / "SLIDE-0329_crop_2048_nuclear.tiff"
NIMBUS_TABLE_PATH = OUTPUT_DIR / "nimbus" / "cell_table_full.csv"

for path in [CROP_DIR, OUTPUT_DIR, FULL_MERGE_PATH, CELL_MASK_PATH, NIMBUS_TABLE_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Crop dir:", CROP_DIR)
print("Full merge:", FULL_MERGE_PATH)
print("Whole-cell mask:", CELL_MASK_PATH)
print("Nuclear mask exists:", NUCLEAR_MASK_PATH.exists())
print("Nimbus table:", NIMBUS_TABLE_PATH)


Crop dir: /mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048
Full merge: /mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/SLIDE-0329_crop_2048_full_merge.ome.tif
Whole-cell mask: /mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff
Nuclear mask exists: True
Nimbus table: /mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/nimbus/cell_table_full.csv


In [2]:
import anndata as ad
import numpy as np
import pandas as pd
import sopa
import spatialdata
import tifffile

from spatialdata import SpatialData, aggregate
from spatialdata.models import Labels2DModel, TableModel

print("spatialdata:", spatialdata.__version__)
print("sopa:", sopa.__version__)
print("pandas:", pd.__version__)
print("tifffile:", tifffile.__version__)


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


spatialdata: 0.7.2
sopa: 2.2.4
pandas: 2.3.3
tifffile: 2026.3.3


## Image Load

Use the proven `sopa.io.ome_tif(...)` path and keep the multiscale image.


In [3]:
image_only_sdata = sopa.io.ome_tif(FULL_MERGE_PATH)
image_key = next(iter(image_only_sdata.images))
full_image = image_only_sdata.images[image_key]
scale0_image = full_image["scale0"]["image"]

print("Imported image key:", image_key)
print("Scale-0 dims:", scale0_image.dims)
print("Scale-0 shape:", scale0_image.shape)
print("First 8 channels:", [str(v) for v in scale0_image.coords["c"].values[:8].tolist()])
print("Coordinate systems:", list(image_only_sdata.coordinate_systems))

image_only_sdata


Imported image key: SLIDE-0329_crop_2048_full_merge
Scale-0 dims: ('c', 'y', 'x')
Scale-0 shape: (24, 2048, 2048)
First 8 channels: ['R1_CY3_AF_I', 'R1_CY5_AF_I', 'R1_CY7_AF_I', 'R1_FITC_AF_I', 'R1_BIT2_RS0584_CY3B', 'R1_BIT3_RS0015_CY5', 'R1_BIT4_RS0083_750', 'R1_DAPI']
Coordinate systems: ['global']


SpatialData object
└── Images
      └── 'SLIDE-0329_crop_2048_full_merge': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128)
with coordinate systems:
    ▸ 'global', with elements:
        SLIDE-0329_crop_2048_full_merge (Images)

## Label Load

Import raster labels directly from the TIFF masks.


In [4]:
cell_mask = tifffile.imread(CELL_MASK_PATH)
print("Cell mask shape:", cell_mask.shape)
print("Cell mask dtype:", cell_mask.dtype)
print("Cell mask max label:", int(cell_mask.max()))

if tuple(cell_mask.shape) != tuple(scale0_image.shape[-2:]):
    raise ValueError(f"Cell mask shape {cell_mask.shape} does not match image shape {tuple(scale0_image.shape[-2:])}.")

cell_labels = Labels2DModel.parse(cell_mask, dims=("y", "x"))
labels = {"cell_labels": cell_labels}

if NUCLEAR_MASK_PATH.exists():
    nuclear_mask = tifffile.imread(NUCLEAR_MASK_PATH)
    print("Nuclear mask shape:", nuclear_mask.shape)
    print("Nuclear mask dtype:", nuclear_mask.dtype)
    print("Nuclear mask max label:", int(nuclear_mask.max()))
    if tuple(nuclear_mask.shape) != tuple(scale0_image.shape[-2:]):
        raise ValueError(f"Nuclear mask shape {nuclear_mask.shape} does not match image shape {tuple(scale0_image.shape[-2:])}.")
    labels["nuclear_labels"] = Labels2DModel.parse(nuclear_mask, dims=("y", "x"))

labels


Cell mask shape: (2048, 2048)
Cell mask dtype: uint32
Cell mask max label: 20752
Nuclear mask shape: (2048, 2048)
Nuclear mask dtype: uint32
Nuclear mask max label: 20672


{'cell_labels': <xarray.DataArray 'image' (y: 2048, x: 2048)> Size: 17MB
 dask.array<array, shape=(2048, 2048), dtype=uint32, chunksize=(2048, 2048), chunktype=numpy.ndarray>
 Coordinates:
   * y        (y) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
   * x        (x) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
 Attributes:
     transform:  {'global': Identity },
 'nuclear_labels': <xarray.DataArray 'image' (y: 2048, x: 2048)> Size: 17MB
 dask.array<array, shape=(2048, 2048), dtype=uint32, chunksize=(2048, 2048), chunktype=numpy.ndarray>
 Coordinates:
   * y        (y) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
   * x        (x) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
 Attributes:
     transform:  {'global': Identity }}

## Nimbus Table Load

Load the merged Nimbus CSV, make the linkage columns explicit, and parse it as a `SpatialData` table linked to `cell_labels`.


In [5]:
nimbus_df = pd.read_csv(NIMBUS_TABLE_PATH)
print("Nimbus table shape:", nimbus_df.shape)
print("Nimbus columns:", nimbus_df.columns.tolist())

if "cell_id" not in nimbus_df.columns:
    raise KeyError("Nimbus table is missing 'cell_id'.")
if "fov" not in nimbus_df.columns:
    raise KeyError("Nimbus table is missing 'fov'.")
if not nimbus_df["cell_id"].is_unique:
    raise ValueError("Nimbus 'cell_id' values are not unique.")

fov_values = sorted(nimbus_df["fov"].astype(str).unique().tolist())
print("Nimbus FOV values:", fov_values)
if len(fov_values) != 1:
    raise ValueError(f"Expected exactly one FOV in Nimbus table, found {fov_values}.")

nimbus_feature_cols = [col for col in nimbus_df.columns if col not in {"cell_id", "fov"}]
non_numeric_cols = [col for col in nimbus_feature_cols if not pd.api.types.is_numeric_dtype(nimbus_df[col])]
if non_numeric_cols:
    raise TypeError(f"Nimbus feature columns must all be numeric. Non-numeric columns: {non_numeric_cols}")

nimbus_obs = nimbus_df[["cell_id", "fov"]].copy()
nimbus_obs["instance_id"] = nimbus_obs["cell_id"].astype(str)
nimbus_obs["region"] = "cell_labels"
nimbus_obs.index = nimbus_obs["instance_id"]

nimbus_var = pd.DataFrame(index=nimbus_feature_cols)
nimbus_adata = ad.AnnData(
    X=nimbus_df[nimbus_feature_cols].to_numpy(),
    obs=nimbus_obs,
    var=nimbus_var,
)
nimbus_table = TableModel.parse(
    nimbus_adata,
    region="cell_labels",
    region_key="region",
    instance_key="instance_id",
)

print("Nimbus table shape after parse:", nimbus_table.shape)
print("Nimbus obs columns:", nimbus_table.obs.columns.tolist())
print("Nimbus var names:", nimbus_table.var_names.tolist())
print("Nimbus region values:", nimbus_table.obs["region"].astype(str).unique().tolist())


Nimbus table shape: (4459, 6)
Nimbus columns: ['cell_id', 'fov', 'SLIDE-0329_1.0.4_R000_DAPI__FINAL_F', 'SLIDE-0329_4.0.4_R000_FITC_GFP-poly-AF488_FINAL_AFR_F', 'SLIDE-0329_4.0.4_R000_Cy3_p19-polyRat_FINAL_AFR_F', 'SLIDE-0329_4.0.4_R000_Cy5_Anxa10-647_FINAL_AFR_F']
Nimbus FOV values: ['SLIDE-0329_crop_2048']
Nimbus table shape after parse: (4459, 4)
Nimbus obs columns: ['cell_id', 'fov', 'instance_id', 'region']
Nimbus var names: ['SLIDE-0329_1.0.4_R000_DAPI__FINAL_F', 'SLIDE-0329_4.0.4_R000_FITC_GFP-poly-AF488_FINAL_AFR_F', 'SLIDE-0329_4.0.4_R000_Cy3_p19-polyRat_FINAL_AFR_F', 'SLIDE-0329_4.0.4_R000_Cy5_Anxa10-647_FINAL_AFR_F']
Nimbus region values: ['cell_labels']


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/spatialdata/models/models.py:1206: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


In [6]:
base_sdata = SpatialData(
    images={"full_image": full_image},
    labels=labels,
    tables={"nimbus_table": nimbus_table},
)

print("Base SpatialData contents:")
print("images:", list(base_sdata.images.keys()))
print("labels:", list(base_sdata.labels.keys()))
print("tables:", list(base_sdata.tables.keys()))
print("coordinate systems:", list(base_sdata.coordinate_systems))

base_sdata


Base SpatialData contents:
images: ['full_image']
labels: ['cell_labels', 'nuclear_labels']
tables: ['nimbus_table']
coordinate systems: ['global']


SpatialData object
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
└── Tables
      └── 'nimbus_table': AnnData (4459, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels)

## Aggregation

Aggregate image intensities directly over `cell_labels`. Keep this result as a separate object.


In [7]:
agg_sdata = aggregate(
    values="full_image",
    by="cell_labels",
    values_sdata=base_sdata,
    by_sdata=base_sdata,
    agg_func="mean",
    table_name="agg_table",
)

agg_table = agg_sdata.tables["agg_table"]

print("Aggregation result:")
print("labels:", list(agg_sdata.labels.keys()))
print("tables:", list(agg_sdata.tables.keys()))
print("agg_table shape:", agg_table.shape)
print("agg_table obs columns:", agg_table.obs.columns.tolist())
print("First 8 aggregated features:", agg_table.var_names[:8].tolist())

agg_sdata


Aggregation result:
labels: ['cell_labels']
tables: ['agg_table']
agg_table shape: (4459, 24)
agg_table obs columns: ['instance_id', 'region']
First 8 aggregated features: ['channel_R1_CY3_AF_I_mean', 'channel_R1_CY5_AF_I_mean', 'channel_R1_CY7_AF_I_mean', 'channel_R1_FITC_AF_I_mean', 'channel_R1_BIT2_RS0584_CY3B_mean', 'channel_R1_BIT3_RS0015_CY5_mean', 'channel_R1_BIT4_RS0083_750_mean', 'channel_R1_DAPI_mean']


SpatialData object
├── Labels
│     └── 'cell_labels': DataArray[yx] (2048, 2048)
└── Tables
      └── 'agg_table': AnnData (4459, 24)
with coordinate systems:
    ▸ 'global', with elements:
        cell_labels (Labels)

## ID Checks

Compare the cell IDs represented in the mask, Nimbus table, and aggregated table.


In [8]:
mask_label_ids = np.unique(cell_mask)
mask_label_ids = mask_label_ids[mask_label_ids > 0]
mask_id_set = set(mask_label_ids.astype(int).tolist())

nimbus_id_set = set(base_sdata.tables["nimbus_table"].obs["instance_id"].astype(int).tolist())
agg_id_set = set(agg_table.obs["instance_id"].astype(int).tolist())

print("Nonzero mask label count:", len(mask_id_set))
print("Nimbus row count:", base_sdata.tables["nimbus_table"].n_obs)
print("Aggregate row count:", agg_table.n_obs)
print("First 10 mask IDs:", sorted(mask_id_set)[:10])
print("First 10 Nimbus IDs:", sorted(nimbus_id_set)[:10])
print("First 10 aggregate IDs:", sorted(agg_id_set)[:10])

print("mask == nimbus:", mask_id_set == nimbus_id_set)
print("mask == aggregate:", mask_id_set == agg_id_set)
print("nimbus == aggregate:", nimbus_id_set == agg_id_set)

print("Nimbus not in mask:", sorted(nimbus_id_set - mask_id_set)[:20])
print("Mask not in Nimbus:", sorted(mask_id_set - nimbus_id_set)[:20])
print("Aggregate not in mask:", sorted(agg_id_set - mask_id_set)[:20])
print("Mask not in aggregate:", sorted(mask_id_set - agg_id_set)[:20])


Nonzero mask label count: 4459
Nimbus row count: 4459
Aggregate row count: 4459
First 10 mask IDs: [85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
First 10 Nimbus IDs: [85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
First 10 aggregate IDs: [85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
mask == nimbus: True
mask == aggregate: True
nimbus == aggregate: True
Nimbus not in mask: []
Mask not in Nimbus: []
Aggregate not in mask: []
Mask not in aggregate: []


In [9]:
nimbus_order = sorted(base_sdata.tables["nimbus_table"].obs["instance_id"].astype(str).tolist(), key=int)
agg_order = sorted(agg_table.obs["instance_id"].astype(str).tolist(), key=int)

print("Row order matches after sorting by instance_id:", nimbus_order == agg_order)
print("First 10 sorted Nimbus IDs:", nimbus_order[:10])
print("First 10 sorted aggregate IDs:", agg_order[:10])


Row order matches after sorting by instance_id: True
First 10 sorted Nimbus IDs: ['85', '86', '87', '88', '89', '90', '91', '92', '93', '94']
First 10 sorted aggregate IDs: ['85', '86', '87', '88', '89', '90', '91', '92', '93', '94']


## Notes

- `base_sdata` is the canonical object for this prototype because it keeps the image, labels, and `nimbus_table` together.
- `agg_sdata` is intentionally separate because `spatialdata.aggregate(...)` returns a reduced object with the aggregated table rather than preserving the preloaded Nimbus table.
- This notebook stays labels-first on purpose. Shapes conversion can come later if needed.

## References

- SpatialData: https://spatialdata.scverse.org/en/stable/index.html
- SpatialData models: https://spatialdata.scverse.org/en/stable/api/models.html
- SpatialData aggregate API: https://spatialdata.scverse.org/en/stable/api/operations.html
- Sopa readers: https://gustaveroussy.github.io/sopa/api/readers/
- Sopa aggregation: https://gustaveroussy.github.io/sopa/api/aggregation/
